# Chapter 5: Cost, Performance, and Model Selection

Estimated time: ~8 hours (including a ~2-2.5 hour conceptual, no-training fine-tuning
subsection near the end).

Prerequisites: Chapter 1 (`agentlib.llm_client`).

Interview category this chapter maps to: cost/latency triage judgment, questions like "GPU
costs doubled, what do you try before adding more GPUs" and "latency jumped from 2s to 12s,
what's the first thing you investigate," both answered here by actually triaging a raw log,
not reciting talking points.

## Concept: token economics, latency, caching, and routing

Input and output tokens are priced separately, and output is almost always more expensive
per token: generation is autoregressive (one token depends on computing every token before
it), while a prompt's input tokens are processed in parallel during prefill. A long system
prompt sent on every call is a fixed tax on every request; a verbose model generating long
answers pays that tax on the expensive side of the ledger.

"Latency" is really four things stacked together, and they have different causes and
different fixes:

| Stage | What it is | What actually helps |
|---|---|---|
| Queueing | Waiting for a worker/GPU to become free | More capacity, backpressure, load shedding |
| Network | Time on the wire, not on any GPU | Regional routing, connection reuse |
| Inference (prefill) | Processing the input prompt | Prompt caching, shorter prompts |
| Generation (decode) | Producing the output, token by token | Shorter outputs, a faster/smaller model |

Conflating these is a common mistake. "Latency is high, add more GPUs" only helps if the
bottleneck is actually queueing. This chapter's break-it section is built around diagnosing
which stage is actually the problem from raw log data.

Prompt caching avoids re-paying prefill cost for content that repeats across calls (a long,
unchanging system prompt, a large retrieved document reused across turns). Model routing is
matching task complexity to model tier: the employee framing from earlier chapters applies
directly here. Don't pay senior-employee rates for a task any junior employee could do
correctly, and don't hand a senior-level task to a junior employee just because they're
cheaper per hour.

## Setup

This chapter uses real tokenization via `tiktoken`. `tiktoken.get_encoding("cl100k_base")`
normally downloads its vocabulary file from `openaipublic.blob.core.windows.net` on first
use, and this repo's build environment can't reach that host (same issue as Hugging Face and
SEC EDGAR, noted in `PROGRESS.md`'s earlier units), so this repo vendors a hash-verified
copy of that exact file in `data/tiktoken_cache/`, pre-seeded into `tiktoken`'s own local
cache directory. `tiktoken` doesn't know the difference: this is genuinely the same,
real, byte-for-byte-identical tokenizer data OpenAI serves, just loaded from a committed
local file instead of a live request. See `PROGRESS.md`'s Unit 6 notes for exactly how this
was verified (SHA-256 hash match against the hash `tiktoken`'s own source code checks for).

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import os
os.environ["TIKTOKEN_CACHE_DIR"] = str(_repo_root / "data" / "tiktoken_cache")

import random

import tiktoken

from agentlib.grading import check
from agentlib import llm_client, synthetic_data

random.seed(42)
encoding = tiktoken.get_encoding("cl100k_base")
print("tiktoken cl100k_base loaded from local cache -- no network call made.")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print("This chapter makes no model calls: the cost and latency work below runs against a\n"
      "synthetic request log and real tokenizer output, so the numbers are reproducible\n"
      "and cost nothing to re-derive. HAS_KEY is printed for consistency with the other\n"
      "chapters, not because anything here branches on it.")

tiktoken cl100k_base loaded from local cache -- no network call made.
LLM_PROVIDER = 'anthropic', HAS_KEY = False
This chapter makes no model calls: the cost and latency work below runs against a
synthetic request log and real tokenizer output, so the numbers are reproducible
and cost nothing to re-derive. HAS_KEY is printed for consistency with the other
chapters, not because anything here branches on it.


## Build: a synthetic request log

`agentlib.synthetic_data.generate_request_log()` produces a realistic-shaped log: Poisson
arrivals, log-normal token counts, and a four-stage latency breakdown per request. It's
seeded, so it's the same log every time this notebook runs.

In [2]:
log = synthetic_data.generate_request_log(n_requests=500, seed=42)

print(f"{len(log)} requests logged.")
print("First request: ", log[0])
avg_latency = sum(r["total_latency_ms"] for r in log) / len(log)
print(f"\nAverage total latency: {avg_latency:.0f}ms -- this is this chapter's 'normal' baseline.")


500 requests logged.
First request:  {'request_id': 'req-00000', 'timestamp': 2.04, 'model': 'haiku', 'input_tokens': 299, 'output_tokens': 227, 'queue_depth': 2, 'queue_time_ms': 73.8, 'network_time_ms': 10.9, 'inference_time_ms': 39.7, 'generation_time_ms': 2619.0, 'total_latency_ms': 2743.4}

Average total latency: 1758ms -- this is this chapter's 'normal' baseline.


### Real tokenization, and its quirks

Real prompts don't tokenize the way you'd guess by counting words. `tiktoken` on a handful
of deliberately quirky examples:

In [3]:
examples = [
    "hello world",
    "Hello, world!",
    "The invoice total is $1,234.56.",
    "def calculate_total(items, discount_code=None):",
    "                                        ",  # lots of whitespace
    "supercalifragilisticexpialidocious",
]

for text in examples:
    tokens = encoding.encode(text)
    pieces = [encoding.decode([t]) for t in tokens]
    print(f"{text!r}")
    print(f"  {len(tokens)} tokens: {pieces}")
    print()


'hello world'
  2 tokens: ['hello', ' world']

'Hello, world!'
  4 tokens: ['Hello', ',', ' world', '!']

'The invoice total is $1,234.56.'
  11 tokens: ['The', ' invoice', ' total', ' is', ' $', '1', ',', '234', '.', '56', '.']

'def calculate_total(items, discount_code=None):'
  9 tokens: ['def', ' calculate', '_total', '(items', ',', ' discount', '_code', '=None', '):']

'                                        '
  1 tokens: ['                                        ']

'supercalifragilisticexpialidocious'
  11 tokens: ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']



A few things worth noticing above, since they're exactly the kind of quirk that surprises
people the first time they check real token counts instead of guessing: punctuation attached
to a word usually splits into its own token; numbers don't tokenize digit-by-digit or as one
whole number, they split at fairly arbitrary boundaries the model learned from training data
frequency; a single long, uncommon word can cost far more tokens than a whole short sentence
(11 tokens for one made-up word above, the same as an entire 6-word sentence with punctuation).
Going the other direction, a long run of pure whitespace collapsed into a single token, not
one-per-space. `cl100k_base` specifically has dedicated merges for repeated whitespace, which
matters a lot for indented code; GPT-2's older tokenizer notoriously didn't have this and
would burn a token per space.

### A latency profiler

"The p99 got worse" is not a diagnosis. Four separable things happen between a request
arriving and a token reaching the user -- it waits in a queue, it crosses the network, the
model processes the prompt, and the model emits tokens -- and a regression lands in exactly
one of them.

Write the decomposition, and report each stage two ways: its total, and its **share of the
total**. The share is the diagnostic half. A stage that grew from 10% to 60% of the time
budget names the culprit; a stage's mean latency in milliseconds, on its own, tells you
something got slower without telling you where to look.

In [ ]:
def profile_latency(requests: list) -> dict:
    '''Decompose a request log into the four stages, as totals AND as shares of the whole.

    Each request carries queue_time_ms, network_time_ms, inference_time_ms and
    generation_time_ms. Return:

        {stage: {"total_ms": <summed across every request>,
                 "pct": <that total as a percentage of all four stages combined>}}

    pct is a share of the grand total, not a mean -- the four shares add up to 100.

    An empty log, or one where every stage is zero, reports zeros rather than dividing by
    zero: a quiet window is a normal thing for a profiler to be handed.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


profile_latency = check("ch05-latency-profile", profile_latency)

In [5]:
profile = profile_latency(log)
print(f"{'stage':20s} {'total_ms':>12s} {'% of total':>12s}")
for stage, stats in profile.items():
    print(f"{stage:20s} {stats['total_ms']:>12.0f} {stats['pct']:>11.1f}%")

stage                    total_ms   % of total
queue_time_ms               41526         4.7%
network_time_ms             12286         1.4%
inference_time_ms           42911         4.9%
generation_time_ms         782329        89.0%


## Break it: three bugs, injected into slices of the same log

For each of the three scenarios below, a slice of the log has a bug injected into it. Try to
diagnose each one from the raw log data alone before reading the reveal; this mirrors how
you'd actually be handed a log excerpt in an interview and asked to figure out what's wrong.

### Scenario 1

Requests `req-00100` through `req-00119` (20 requests) look like this, a slice of the raw
log, nothing else:

In [6]:
def inject_duplicate_calls(log: list, start: int, count: int) -> list:
    '''The bug: a retry path that isn't idempotent double-executes the call, so both
    input and output tokens are ~2x what a normal request of this kind costs -- but it's
    still logged as a single request.'''
    injected = [dict(r) for r in log]
    for i in range(start, start + count):
        injected[i] = dict(injected[i])
        injected[i]["input_tokens"] *= 2
        injected[i]["output_tokens"] *= 2
        injected[i]["inference_time_ms"] = round(injected[i]["input_tokens"] * 0.175, 1)
        injected[i]["generation_time_ms"] = round(injected[i]["output_tokens"] * 11.5, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_1_log = inject_duplicate_calls(log, start=100, count=20)

for r in scenario_1_log[95:105]:
    print(f"{r['request_id']}  input_tokens={r['input_tokens']:5d}  output_tokens={r['output_tokens']:4d}")


req-00095  input_tokens=  285  output_tokens=  88
req-00096  input_tokens=  484  output_tokens= 165
req-00097  input_tokens=  644  output_tokens= 142
req-00098  input_tokens=  179  output_tokens= 207
req-00099  input_tokens=  490  output_tokens= 192
req-00100  input_tokens= 2074  output_tokens= 276
req-00101  input_tokens= 2086  output_tokens= 244
req-00102  input_tokens=  568  output_tokens= 150
req-00103  input_tokens=  552  output_tokens= 172
req-00104  input_tokens= 1576  output_tokens= 172


Diagnose before reading on. What would you check first, and what does it tell you?

Write your diagnosis in the next cell before moving on. The worked answer is in `solutions/reference/ch05.py`; read it after yours passes, not before.

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_DUPLICATE_CALL_DIAGNOSIS = """Replace this with your diagnosis."""


MY_DUPLICATE_CALL_DIAGNOSIS = check("ch05-diagnose-duplicate-calls", MY_DUPLICATE_CALL_DIAGNOSIS)

In [8]:
def call_with_idempotency(request_id: str, already_processed: set, fn):
    '''The fix: check an idempotency key BEFORE executing, so a duplicate trigger (a retry,
    a double-submit) short-circuits instead of re-running the expensive call.'''
    if request_id in already_processed:
        return "skipped -- already processed this request_id"
    already_processed.add(request_id)
    return fn()


processed = set()
print("Simulating the same trigger firing twice for req-00100:")
print(" ", call_with_idempotency("req-00100", processed, lambda: "executed the real call"))
print(" ", call_with_idempotency("req-00100", processed, lambda: "executed the real call"))
print("\nOnly one real execution happened -- the second trigger was recognized and skipped")
print("before it could double the token volume.")


Simulating the same trigger firing twice for req-00100:
  executed the real call
  skipped -- already processed this request_id

Only one real execution happened -- the second trigger was recognized and skipped
before it could double the token volume.


### Scenario 2

Requests `req-00200` through `req-00229` (30 requests, one growing conversation):

In [9]:
def inject_unbounded_context_growth(log: list, start: int, count: int, growth_per_turn: int = 180) -> list:
    '''The bug: a conversational agent that appends every prior turn to the prompt with no
    truncation -- input_tokens grows roughly linearly across a session instead of staying
    in its normal range.'''
    injected = [dict(r) for r in log]
    for offset in range(count):
        i = start + offset
        injected[i] = dict(injected[i])
        injected[i]["input_tokens"] = 150 + offset * growth_per_turn
        injected[i]["inference_time_ms"] = round(injected[i]["input_tokens"] * 0.175, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_2_log = inject_unbounded_context_growth(log, start=200, count=30)

for r in scenario_2_log[200:230:5]:
    print(f"{r['request_id']}  input_tokens={r['input_tokens']:5d}  inference_time_ms={r['inference_time_ms']:7.1f}")


req-00200  input_tokens=  150  inference_time_ms=   26.2
req-00205  input_tokens= 1050  inference_time_ms=  183.8
req-00210  input_tokens= 1950  inference_time_ms=  341.2
req-00215  input_tokens= 2850  inference_time_ms=  498.7
req-00220  input_tokens= 3750  inference_time_ms=  656.2
req-00225  input_tokens= 4650  inference_time_ms=  813.8


Diagnose before reading on.

Write your diagnosis in the next cell before moving on. The worked answer is in `solutions/reference/ch05.py`; read it after yours passes, not before.

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_CONTEXT_GROWTH_DIAGNOSIS = """Replace this with your diagnosis."""


MY_CONTEXT_GROWTH_DIAGNOSIS = check("ch05-diagnose-context-growth", MY_CONTEXT_GROWTH_DIAGNOSIS)

In [11]:
def apply_context_window_policy(history: list, max_turns: int = 6) -> list:
    '''The fix: cap how much conversation history gets re-sent, instead of appending
    forever.'''
    return history[-max_turns:]


long_history = [f"turn {i}: ..." for i in range(30)]
print(f"Full history: {len(long_history)} turns")
print(f"After the window policy: {len(apply_context_window_policy(long_history))} turns re-sent per call")
print("\nInput tokens per call now stay roughly flat regardless of how long the conversation runs,")
print("instead of growing without bound.")


Full history: 30 turns
After the window policy: 6 turns re-sent per call

Input tokens per call now stay roughly flat regardless of how long the conversation runs,
instead of growing without bound.


### Scenario 3

Requests `req-00300` through `req-00329` (30 requests). This is the classic interview
framing verbatim: "latency jumped from ~2s to ~12s, what's the first thing you
investigate?"

In [12]:
def inject_queueing_spike(log: list, start: int, count: int, peak_queue_depth: int = 42) -> list:
    '''The bug: a burst of concurrent traffic (a batch job, a retry storm, a traffic spike)
    overwhelms available capacity. Nothing about any individual request is unusual -- its
    own input/output tokens are normal -- but queue_depth spikes, so queue_time_ms dominates
    total latency.'''
    injected = [dict(r) for r in log]
    for offset in range(count):
        i = start + offset
        injected[i] = dict(injected[i])
        # Ramp queue depth up and back down across the window (a bursty spike, not a step).
        progress = offset / (count - 1)
        depth = int(peak_queue_depth * (1 - abs(2 * progress - 1)))
        injected[i]["queue_depth"] = depth
        injected[i]["queue_time_ms"] = round(depth * 270.0, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_3_log = inject_queueing_spike(log, start=300, count=30)

baseline_latency = sum(r["total_latency_ms"] for r in log[270:300]) / 30
spike_latency = max(r["total_latency_ms"] for r in scenario_3_log[300:330])
print(f"Baseline total_latency_ms (pre-spike window): {baseline_latency:.0f}ms")
print(f"Peak total_latency_ms (inside the spike):      {spike_latency:.0f}ms")
print()
for r in scenario_3_log[300:330:4]:
    print(
        f"{r['request_id']}  queue_depth={r['queue_depth']:3d}  "
        f"queue_time_ms={r['queue_time_ms']:7.1f}  input_tokens={r['input_tokens']:5d}  "
        f"total_latency_ms={r['total_latency_ms']:8.1f}"
    )


Baseline total_latency_ms (pre-spike window): 1696ms
Peak total_latency_ms (inside the spike):      12428ms

req-00300  queue_depth=  0  queue_time_ms=    0.0  input_tokens=  544  total_latency_ms=  1231.3
req-00304  queue_depth= 11  queue_time_ms= 2970.0  input_tokens=  306  total_latency_ms=  5159.5
req-00308  queue_depth= 23  queue_time_ms= 6210.0  input_tokens=  430  total_latency_ms=  8171.7
req-00312  queue_depth= 34  queue_time_ms= 9180.0  input_tokens=  392  total_latency_ms= 10078.5
req-00316  queue_depth= 37  queue_time_ms= 9990.0  input_tokens=  388  total_latency_ms= 11182.0
req-00320  queue_depth= 26  queue_time_ms= 7020.0  input_tokens=  726  total_latency_ms=  8016.8
req-00324  queue_depth= 14  queue_time_ms= 3780.0  input_tokens=  195  total_latency_ms=  5619.3
req-00328  queue_depth=  2  queue_time_ms=  540.0  input_tokens=  412  total_latency_ms=  2136.6


Diagnose before reading on. Given the four-stage breakdown from the profiler above,
which stage would you check first when someone reports "latency went from 2s to 12s"?

Write your diagnosis in the next cell before moving on. The worked answer is in `solutions/reference/ch05.py`; read it after yours passes, not before.

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_QUEUEING_DIAGNOSIS = """Replace this with your diagnosis."""


MY_QUEUEING_DIAGNOSIS = check("ch05-diagnose-queueing", MY_QUEUEING_DIAGNOSIS)

In [14]:
def summarize_stage_share(requests: list) -> dict:
    '''The diagnostic step itself: decompose a slice back into per-stage totals, the same
    way profile_latency() did for the whole log, so a spike shows up as a shift in *which*
    stage dominates rather than just a bigger total.'''
    stages = ["queue_time_ms", "network_time_ms", "inference_time_ms", "generation_time_ms"]
    totals = {stage: sum(r[stage] for r in requests) for stage in stages}
    grand_total = sum(totals.values())
    return {stage: totals[stage] / grand_total * 100 for stage in stages}


print("Stage share, normal window (requests 270-299):")
for stage, pct in summarize_stage_share(log[270:300]).items():
    print(f"  {stage:20s} {pct:5.1f}%")

print("\nStage share, spike window (requests 300-329):")
for stage, pct in summarize_stage_share(scenario_3_log[300:330]).items():
    print(f"  {stage:20s} {pct:5.1f}%")


Stage share, normal window (requests 270-299):
  queue_time_ms          5.5%
  network_time_ms        1.3%
  inference_time_ms      5.8%
  generation_time_ms    87.4%

Stage share, spike window (requests 300-329):
  queue_time_ms         76.5%
  network_time_ms        0.4%
  inference_time_ms      1.3%
  generation_time_ms    21.8%


The fix for a genuine queueing spike is capacity and backpressure, not per-request
optimization: there's no prompt to shorten or model to swap here, since no individual
request is doing anything wrong. A simple, honest fix to demonstrate: shed load past a
depth threshold instead of letting queue time grow unbounded, so latency degrades
predictably (a fast rejection) instead of catastrophically (every request waiting behind an
ever-growing line).

In [15]:
def admit_with_backpressure(queue_depth: int, max_queue_depth: int = 15) -> str:
    '''The fix: shed load past a depth threshold instead of letting every request queue
    indefinitely behind an unbounded backlog.'''
    if queue_depth > max_queue_depth:
        return "rejected -- 503, retry with backoff (protects requests already in flight)"
    return "admitted"


print("Without backpressure, every one of these would queue and add to total_latency_ms:")
for depth in [5, 12, 25, 38]:
    print(f"  queue_depth={depth:3d}  ->  {admit_with_backpressure(depth)}")


Without backpressure, every one of these would queue and add to total_latency_ms:
  queue_depth=  5  ->  admitted
  queue_depth= 12  ->  admitted
  queue_depth= 25  ->  rejected -- 503, retry with backoff (protects requests already in flight)
  queue_depth= 38  ->  rejected -- 503, retry with backoff (protects requests already in flight)


## Optimize: prompt caching and model routing

Two concrete levers, demonstrated against the same log rather than described abstractly.

### Prompt caching

If a large chunk of the prompt repeats across calls (a long system prompt, a retrieved
document reused across several turns), paying full prefill cost for it every single call is
pure waste. A cache hit skips re-processing the cached prefix; only the new suffix (plus
generation) still costs full price. This is exactly Jack's situation from Chapter 2's stale-
cache scenario, but pointed at cost instead of correctness: the same "don't redo work that
didn't change" instinct, applied to token spend.

The cost model is yours to write. Two things it has to get right, and both are places where
a plausible simplification quietly understates the bill: input and output tokens are priced
differently (output is the expensive half, typically 4-5x input), and cached tokens are
*cheap*, not free -- a cache read still bills, just at a steep discount.

In [ ]:
def estimate_cost_with_caching(input_tokens: int, output_tokens: int, cached_prefix_tokens: int,
                                cache_hit: bool, input_price=3.0, output_price=15.0, cache_price=0.30) -> dict:
    '''Per-million-token prices in dollars (illustrative, Sonnet-class ballpark).

    On a cache MISS you pay full input price for every input token, prefix included, and the
    cached_prefix_tokens figure is irrelevant.

    On a cache HIT the prefix is billed at cache_price instead of input_price, and only the
    remaining (input_tokens - cached_prefix_tokens) count as fresh input.

    Output is always billed at output_price -- note it is 5x the input price here, which is
    why pricing both sides the same understates exactly the verbose responses you most want
    a cost model to flag.

    Return {"cache_hit": ..., "fresh_input_tokens": ..., "cost_usd": round(cost, 6)}.
    Prices are per MILLION tokens.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


estimate_cost_with_caching = check("ch05-token-cost", estimate_cost_with_caching)

In [17]:
# A long, reused system prompt (2,000 tokens) plus a short per-call question (50 tokens).
no_cache = estimate_cost_with_caching(input_tokens=2050, output_tokens=300, cached_prefix_tokens=2000, cache_hit=False)
with_cache = estimate_cost_with_caching(input_tokens=2050, output_tokens=300, cached_prefix_tokens=2000, cache_hit=True)

print("Without caching:", no_cache)
print("With caching:   ", with_cache)
savings_per_call = no_cache["cost_usd"] - with_cache["cost_usd"]
print(f"\nSavings per call: ${savings_per_call:.6f}  ->  over 10,000 calls: ${savings_per_call * 10000:.2f}")

Without caching: {'cache_hit': False, 'fresh_input_tokens': 2050, 'cost_usd': 0.01065}
With caching:    {'cache_hit': True, 'fresh_input_tokens': 50, 'cost_usd': 0.00525}

Savings per call: $0.005400  ->  over 10,000 calls: $54.00


### Model routing

Route each request to the cheapest model that can still do the task correctly, instead of
sending everything to the strongest (most expensive) model by default. This only works if
routing decisions are based on something real about the task, not a guess.

Two independent reasons to escalate, and the second is the one that gets dropped. Length is
easy to see and easy to code. Tool use is the reason that actually costs money when you get
it wrong -- a multi-step tool loop is where a weak model's mistakes compound -- and the
requests that need it are frequently short.

In [ ]:
def route_request(prompt: str, requires_tool_use: bool = False) -> str:
    '''A simple, legible routing policy.

    Send a request to the strong tier if EITHER it needs multi-step tool use OR it is longer
    than 80 words (counted in words, not characters). Otherwise the cheap/fast tier.
    Exactly 80 words is not yet over the line.

    Return an actual model id -- llm_client.STRONG_MODELS[llm_client.LLM_PROVIDER] or
    llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER] -- not a tier label, since the caller
    passes this straight to call_model().

    A real production router would likely use a small classifier or a cheap model call to
    make this decision; the policy itself is the point here, not the classifier.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


route_request = check("ch05-router", route_request)

In [19]:
test_requests = [
    ("What's the capital of France?", False),
    ("Summarize this 40-page contract, flag every clause that deviates from our standard "
     "template, and cross-reference each flagged clause against the three most similar "
     "clauses from our last 20 signed contracts.", True),
    ("Format this as a bulleted list.", False),
]

for prompt, needs_tools in test_requests:
    model = route_request(prompt, needs_tools)
    print(f"[{model}]  {prompt[:60]}{'...' if len(prompt) > 60 else ''}")

[claude-haiku-4-5-20251001]  What's the capital of France?
[claude-sonnet-5]  Summarize this 40-page contract, flag every clause that devi...
[claude-haiku-4-5-20251001]  Format this as a bulleted list.


## Concept: fine-tuning, without training anything

This subsection is deliberately conceptual. No GPU, no training run, nothing to execute.
The goal is being able to speak accurately about *when fine-tuning is the right lever at
all*, since in agent-engineering work the far more common answer is "better prompting,
better retrieval, or better tool design," not "fine-tune a model." Interviewers probing this
area are usually checking whether a candidate reaches for fine-tuning reflexively (a red
flag) or only after ruling out cheaper, faster levers.

#### When fine-tuning is (and isn't) the right lever

Prompting and RAG changes ship in minutes and are reversible; a fine-tune is a
multi-hour-to-multi-day training run that produces a new artifact to version, serve, and
eventually retrain. Reach for fine-tuning when the need is a *stable, repeated* behavior
change that prompting can't reliably produce: a consistent output format at high volume, a
narrow domain vocabulary, a specific tool-calling style, not for a one-off task or something
still being iterated on. If better retrieval or a clearer prompt would fix it, that's almost
always the cheaper and faster fix to try first.

#### LoRA and QLoRA

Full fine-tuning updates every model weight, which is expensive and produces a full-size new
copy of the model per task. LoRA (Low-Rank Adaptation; Hu et al., 2021) instead freezes the
original weights and trains a small pair of low-rank matrices injected alongside them: far
fewer trainable parameters, a much smaller artifact to store per task, and the base model
can be shared across many LoRA adapters. QLoRA (Dettmers et al., 2023) combines this with
quantizing the frozen base model to 4-bit precision during training, cutting the GPU memory
needed enough to fine-tune large models on a single consumer-class GPU.

#### RLHF, and its role in agents specifically

Reinforcement Learning from Human Feedback (Christiano et al., 2017; applied to
instruction-following at scale in Ouyang et al., 2022's InstructGPT work) is how base
language models get shaped into models that follow instructions and prefer helpful, honest,
harmless responses. It's part of how the underlying model you call was trained, not
something an agent engineer typically does themselves. Where it becomes directly relevant to
*this* course's scope: the reward signal in RLHF (or the newer RLAIF/constitutional-AI
variants) is a judgment about response quality, and that same "have a model judge a
response" pattern is exactly what Chapter 3's faithfulness scoring and an LLM-as-judge eval
harness do at inference time. Understanding RLHF conceptually is what lets you recognize
that connection instead of treating them as two unrelated topics.

## Recap

This chapter covered: token economics (why output tokens cost more than input tokens);
decomposing latency into queueing, network, inference, and generation instead of
treating it as one number; diagnosing three real bugs from raw log data alone, a
duplicate-call bug, unbounded context growth, and a queueing spike, each with a working fix;
optimization levers (prompt caching, model routing) demonstrated against real cost math
rather than described abstractly; and a conceptual (no-training) tour of fine-tuning, LoRA/
QLoRA, and RLHF, including how RLHF's core mechanic connects back to Chapter 3's evaluation
work.

Next: Chapter 6 moves from a single agent to a small team, Jack, Bob, and Mike working
together, and the coordination patterns (and coordination failures) that come with more than
one agent.

## Interview drill

Answer each of these on your own, in writing, out loud, or both, before checking
`solutions/ch05_cost_performance_model_selection_answers.md`.


1. Cold diagnosis. You're told: "our per-request GPU cost doubled overnight, no model
or traffic-volume change was deployed." Walk through what you'd check, in order, and why
each check comes before the next.

2. Cold diagnosis. You're told: "p99 latency jumped from about 2 seconds to about 12
seconds starting this morning." Using the four-stage latency breakdown from this chapter,
what's the first thing you'd look at, and what would you expect to see if your hypothesis is
right vs. wrong?

3. Judgment call. A teammate proposes fine-tuning a model to fix a chatbot that keeps
giving answers in the wrong tone for your brand. What would you ask before agreeing that
fine-tuning (rather than prompting) is the right lever here?

4. Conceptual. Where does RLHF actually show up in an agent engineer's day-to-day work,
if at all? What's the closest thing to RLHF's core mechanic that shows up elsewhere in this
course?